In [1]:
import os
import re
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)          # searches cwd and upward
print(f"working directory   : {os.getcwd()}")
print(f"found .env at       : {env_path or 'NOT FOUND'}")

loaded = load_dotenv(env_path) if env_path else False
print(f"load_dotenv() said  : {loaded}")

if "GOOGLE_API_KEY" in os.environ:
    key = os.environ["GOOGLE_API_KEY"]
    print(f"key loaded          : {key[:6]}...{key[-4:]}   ({len(key)} characters)")
else:
    print("GOOGLE_API_KEY is NOT set — see troubleshooting below")

working directory   : e:\4-1 AI\my-ai-repo\repo
found .env at       : e:\4-1 AI\my-ai-repo\repo\.env
load_dotenv() said  : True
key loaded          : AQ.Ab8...lGow   (53 characters)


In [2]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

In [ ]:
MODEL = "gemini-3.5-flash-lite"

r = client.models.generate_content(
    model=MODEL,
    contents="Reply with exactly one word: hello",
    config=types.GenerateContentConfig(
        temperature=0,
        max_output_tokens=5,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    ),
)
print(r.text)
print("tokens — in:", r.usage_metadata.prompt_token_count,
      "out:", r.usage_metadata.candidates_token_count)

In [5]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

URL = ("https://raw.githubusercontent.com/yjin-chae/"
       "LLMs-for-text-classification/main/data/original/semeval_2016.csv")

df = pd.read_csv(URL, encoding="latin-1")           # the file has non-UTF-8 bytes
df = df[["Tweet", "Target", "Stance"]].reset_index(drop=True)

print(df.shape)
print("\nTarget x Stance:")
print(pd.crosstab(df["Target"], df["Stance"], margins=True))

(1691, 3)

Target x Stance:
Stance           AGAINST  FAVOR  NONE   All
Target                                     
Donald Trump         299    148   260   707
Hillary Clinton      565    163   256   984
All                  864    311   516  1691


In [6]:
strat = df["Target"] + "|" + df["Stance"]

train_df, test_df = train_test_split(df, test_size=0.30, random_state=42, stratify=strat)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# 200-tweet held-out lab set. Everything below is scored on this.
lab_df = test_df.sample(n=200, random_state=42).reset_index(drop=True)

print(f"train {len(train_df)}   test {len(test_df)}   lab set {len(lab_df)}")
print("\nlab set Target x Stance:")
print(pd.crosstab(lab_df["Target"], lab_df["Stance"]))

train 1183   test 508   lab set 200

lab set Target x Stance:
Stance           AGAINST  FAVOR  NONE
Target                               
Donald Trump          42     20    31
Hillary Clinton       59     14    34


In [7]:
LABELS = ["AGAINST", "FAVOR", "NONE"]

# The paper's exact zero-shot prompt, copied verbatim from prompt3 in the
# replication package. Note the trailing "\n\n" — the tweet gets appended after.
PROMPT = (
    "These statement contains a TARGET and a STANCE. The target is a politician "
    "and the stance represents the attitude expressed about them. The target "
    "options are Trump or Clinton and stance options are Favor, Against or None. "
    "Provide the answer in the following format: {TARGET, STANCE}\n\n"
)

In [8]:
tweet = "Some example tweet #SemST"
cleaned = tweet.replace("#SemST", "").strip()          # → "Some example tweet"

In [9]:
row = lab_df.iloc[0]
tweet = row["Tweet"].replace("#SemST", "").strip()     # clean it, same as the paper does

print(PROMPT + tweet)

These statement contains a TARGET and a STANCE. The target is a politician and the stance represents the attitude expressed about them. The target options are Trump or Clinton and stance options are Favor, Against or None. Provide the answer in the following format: {TARGET, STANCE}

@3_Card_Monty : I believe the title would be "First Lord". Let THAT sink in!! #PJNET


In [10]:
for k in range(5):
    row = lab_df.iloc[k]
    tweet = row["Tweet"].replace("#SemST", "").strip()
    r = client.models.generate_content(
        model=MODEL,
        contents=PROMPT + tweet,
        config=types.GenerateContentConfig(
            temperature=0,
            max_output_tokens=200,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        ),
    )
    reply = r.text
    print(f"[{k}] true=({row['Target']:15s}, {row['Stance']:8s})   "
          f"reply={reply.strip()[:80]!r}   ")

AttributeError: 'NoneType' object has no attribute 'strip'

In [ ]:
import time

def annotate_corpus(tweets, prompt=PROMPT, temperature=0):
    """Send every tweet to the model, one at a time. Returns a list of raw replies."""
    replies = []
    for tw in tweets:
        tweet = tw.replace("#SemST", "").strip()
        r = client.models.generate_content(
            model=MODEL,
            contents=prompt + tweet,
            config=types.GenerateContentConfig(
                temperature=temperature,
                max_output_tokens=200,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
            ),
        )
        replies.append(r.text)
        if len(replies) % 25 == 0:
            print(f"  {len(replies)}/{len(tweets)} done")
        time.sleep(6)   # 10 requests/minute, safely under the 15/min free-tier cap
    return replies

raw_zs = annotate_corpus(lab_df["Tweet"].tolist())
print(f"\n{len(raw_zs)} replies received. First one:\n{raw_zs[0]}")